# LELA60331 Week 3 Seminar

This week's seminar has two parts:

| Part | Topic |
|---|---|
| 1 | More n-gram language models: Python practice with files and strings, perplexity, trigram entropy, and language models for words |
| 2 | Vector-based models of word meaning |

Part 1 carries on from the Week 2 seminar. Part 2 introduces a new idea: representing the meaning of a word as a vector.

Each **Activity** is short: replace every `???` with your own code and run the cell. Parts marked ⭐ **Challenge** are optional. Apart from the `math` library, which is explained when we first use it, this sheet only uses Python from the Week 2 and Week 3 Python workbooks (LELA60341).

---
# Part 1 · More n-gram language models

## Setting up

The first code cell repeats the set-up from the Week 2 seminar. It downloads *Crime and Punishment*, keeps the letters a–z and the spaces of chapter one, and counts the characters (unigrams) and the pairs of neighbouring characters (bigrams). Run it before anything else.

In [ ]:
# download the book from Project Gutenberg (the ! runs a command outside Python)
!wget https://www.gutenberg.org/files/2554/2554-0.txt
# read the whole file into one string
with open('2554-0.txt') as fp:
    c_and_p = fp.read()
# find where chapter one starts and ends: from just after the heading 'CHAPTER I'
# up to the heading 'CHAPTER II' (Activity 2 looks at this more closely)
start = c_and_p.find('CHAPTER I\n') + len('CHAPTER I\n')
end = c_and_p.find('CHAPTER II\n')
# select chapter one and convert it to lower case
raw = c_and_p[start:end].lower()
# keep only the letters a-z and the space, using a for loop
allowed = 'abcdefghijklmnopqrstuvwxyz '
kept = []
for ch in raw:
    if ch in allowed:
        kept.append(ch)
chapter_one = ''.join(kept)       # glue the kept characters back into one string

# count each character (unigrams) and each pair of neighbouring characters (bigrams), as in Week 2
unigrams = {}
for ch in chapter_one:
    unigrams[ch] = unigrams.get(ch, 0) + 1
total_unigrams = len(chapter_one)

bigrams = {}
total_bigrams = len(chapter_one) - 1
for i in range(total_bigrams):
    pair = chapter_one[i:i+2]
    bigrams[pair] = bigrams.get(pair, 0) + 1

# math is a library of mathematical functions that comes with Python. 'import math' loads it,
# and then math.log(x, 2) gives the logarithm of x to base 2.
import math

### Python practice: files and strings

Before going further, let's practise some Python from the Week 2 and Week 3 Python workbooks (LELA60341) on the text we just downloaded. Each **Activity** is short: replace every `???` with your own code and run the cell. Parts marked ⭐ **Challenge** are optional.

**Activity 1 (reading a file line by line)**

Above, we read the whole file into one long string with `fp.read()`. `fp.readlines()` instead gives a **list** of lines.

Fill in the blank to print how many lines the file has, then run the cell to see lines 100 to 109.

In [ ]:
with open('2554-0.txt') as fp:
    lines = fp.readlines()

print('Number of lines:', ???)    # hint: len() gives the length of a list

for line in lines[100:110]:
    print(line.strip())

**Activity 2 (checking where chapter one starts and ends)**

In the first code cell, `.find()` gave us the positions `start` and `end`. Let's check them by looking at the text either side.

The first `print` shows the 60 characters just **after** `start`. Fill in the slice for the 60 characters just **before** `end`.

In [ ]:
print(start, end)
print(c_and_p[start:start+60])    # the 60 characters after start
print('-----')
print(c_and_p[???:???])           # the 60 characters before end

`.find()` only gives the **first** match. The cell below uses a `while` loop to find **every** `'CHAPTER II\n'` in the book. Run it, then think about these questions:

* Why are there several matches? Why does our code in the first code cell still pick the right one?
* Why do we search for `'CHAPTER II\n'` (with a newline) and not just `'CHAPTER II'`? (What else starts with those letters?)

In [ ]:
positions = []
pos = c_and_p.find('CHAPTER II\n')
while pos != -1:                                   # -1 means "not found"
    positions.append(pos)
    rest = c_and_p[pos + 1:]                       # the text just after this match
    next_pos = rest.find('CHAPTER II\n')
    if next_pos == -1:
        pos = -1                                   # no more matches: the loop will stop
    else:
        pos = pos + 1 + next_pos                   # turn it back into a position in the whole book

for p in positions:
    print(p, c_and_p[p:p+40])

**Activity 3 (improving the cleaning)**

The first line of the cell below prints the start of `chapter_one`. The newline character `'\n'` is not in `allowed`, so it was thrown away, and the last word of each line got stuck to the first word of the next.

Fill in the blank so that every newline is replaced by a space **before** the loop. Then compare the number of words before and after. Why are there more words afterwards?

In [ ]:
print(chapter_one[:300])

raw2 = c_and_p[start:end].lower()
raw2 = raw2.replace(???, ' ')     # turn every newline into a space

kept = []
for ch in raw2:
    if ch in allowed:
        kept.append(ch)
clean = ''.join(kept)

print(len(chapter_one.split()), 'words before')
print(len(clean.split()), 'words after')

## Saving counts to a file

**Activity 4 (saving counts to a file)**

Counting takes time on large texts, so we often save the counts to a file.

1. Fill in the blank to write one line per character in the form `character<TAB>count`. `fp.write()` only accepts strings, so the count has to be converted with `str()`. `'\t'` is a tab and `'\n'` a newline.
2. The second part reads the file back and shows the first 5 lines. How many lines should the file have, and why?

In [ ]:
# 1. write
with open('char_unigrams.tsv', 'w') as fp:
    for char, count in unigrams.items():
        fp.write(char + '\t' + ??? + '\n')

# 2. read back
with open('char_unigrams.tsv') as fp:
    lines = fp.readlines()
print(len(lines), 'lines')
for line in lines[:5]:
    print(line.rstrip())

⭐ **Challenge (optional):** rebuild the counts dictionary from `lines`. `split('\t')` separates the character from the count, and `int()` turns the count back into a number.

In [ ]:
loaded = {}
for line in lines:
    fields = line.rstrip().split('\t')     # e.g. ['e', '2190']
    loaded[fields[0]] = ???

print(loaded == unigrams)

## Perplexity

In the Week 2 seminar we worked out the total surprisal of a sentence under the character bigram model. Here is that cell again. Run it, because Activity 5 uses `sentence` and `log_prob`.

In [ ]:
import math
# math.log(x, 2) is the logarithm of x to base 2
# The sentence is a string, so each element is a character.
# It starts with a space, which marks the start of the first word.
sentence = " the cat on the mat and ate his dinner"
log_prob = 0.0
prob = 1.0
for i in range(len(sentence) - 1):
    pair = sentence[i:i+2]
    p = bigrams.get(pair, 0) / unigrams[pair[0]]      # P(next character | this character)
    log_prob += -math.log(p, 2)
    prob *= p
log_prob
#prob

Raw log probability of a text grows more negative the longer the text is, so it can't fairly compare texts or models. **Perplexity** fixes this by averaging per token and putting the result on an intuitive scale.

### Cross entropy

The average negative log probability the model gave the true tokens:

$$
H = -\frac{1}{N} \sum_{i=1}^{N} \log p(x_i \mid x_{<i})
$$

### Perplexity

The exponential of cross entropy, using the same log base:

$$
\text{PPL} = e^{H}
$$

It's the **effective number of equally likely choices** the model was torn between at each step:

- **PPL = 1:** the model was certain and right every time.
- **PPL = 20:** the model was as uncertain as picking uniformly among 20 options.

> **Note:** Values are only comparable across models that share a tokenizer and are evaluated on the same text.

**Activity 5 (perplexity)**

`log_prob` from the sentence cell above is the **total** surprisal of the sentence. To compare sentences of different lengths we use the **average** surprisal per bigram (the cross-entropy), and then "perplexity":

* cross-entropy = `log_prob` ÷ number of bigrams
* perplexity = 2 to the power of the cross-entropy (`2 ** x` in Python)

1. Fill in the two blanks and run the cell.
2. Now change `sentence` in the cell above to `" sat the on mat cat the"` (the same words in a jumbled order), run that cell again, then run this one again. Which sentence is more surprising to the model? Why?

In [ ]:
n_bigrams = len(sentence) - 1
cross_entropy = ???      # log_prob divided by n_bigrams
perplexity = ???         # 2 to the power of cross_entropy
print(sentence, '|', round(cross_entropy, 2), '|', round(perplexity, 2))

## Entropy from a trigram model

In the Week 2 seminar we estimated the entropy of English from the character bigram model with the cell below. Run it again so that you can compare it with the trigram estimate.

In [ ]:
H = 0.0
for pair, count in bigrams.items():
    p_pair = count / total_bigrams          # P(x, y): how often this pair occurs
    p_next = count / unigrams[pair[0]]      # P(y | x): how often y follows x
    H += -p_pair * math.log(p_next, 2)
H

**Activity 6 (trigram entropy)**

Estimate entropy from a trigram character model. First of all you will need to rewrite the code to extract trigram counts.

In [ ]:
# Count each sequence of three neighbouring characters (a trigram)
trigrams = {}
total_trigrams = ???
for i in range(total_trigrams):
    triple = chapter_one[???]
    trigrams[triple] = trigrams.get(triple, 0) + 1

Next you will need to use these to calculate the trigram probabilities and combine to calculate the entropy.

In [ ]:
H = 0.0
for triple, count in trigrams.items():
    p_triple = count / ???                  # P(x, y, z)
    p_next = count / bigrams[???]           # P(z | x, y)
    H += -p_triple * math.log(p_next, 2)
H

## Language modelling for words

We are now going to switch to building and using word-based language models. Instead of a string of characters we need a **list of words**. We tidy the text with the string methods from the Week 2 Python workbook (`lower()`, `replace()`) and then use `split()`, which breaks a string at whitespace into a list of words and ignores repeated spaces.

In [ ]:
# c_and_p (the whole book), start, end and allowed were set in the first code cell
raw = c_and_p[start:end].lower()
# line breaks become spaces
raw = raw.replace('\n', ' ')
# mark the end of each sentence with the special token 'eol'
raw = raw.replace('. ', ' eol ')
# keep only a-z and space, as before
kept = []
for ch in raw:
    if ch in allowed:
        kept.append(ch)
# glue the characters back together, then split at whitespace to get a list of words
chapter_one = ''.join(kept).split()

With the text in this new format, the same counting code works. The only difference is that each element is now a word rather than a character.

In [ ]:
unigrams = {}
for word in chapter_one:
    unigrams[word] = unigrams.get(word, 0) + 1

total_unigrams = len(chapter_one)

In [ ]:
# A word bigram is two neighbouring words. We join them with a space
# so that the pair can be used as one dictionary key, e.g. 'the man'
bigrams = {}
total_bigrams = len(chapter_one) - 1
for i in range(total_bigrams):
    pair = " ".join(chapter_one[i:i+2])
    bigrams[pair] = bigrams.get(pair, 0) + 1

In [ ]:
unigrams

In [ ]:
bigrams

**Activity 7 (counting words)**

`chapter_one` is now a **list** of words.

1. `len()` gives the number of word *tokens*. Use `set()` to get the number of *different* words (types).
2. We replaced each full stop with the word `eol`. Use the list method `.count()` to count the sentences.
3. Fill in the blank to work out the average length of a word in characters. The loop skips the `eol` tokens with `continue`.

In [ ]:
# 1.
n_tokens = len(chapter_one)
n_types = ???            # hint: len() of a set of the words
print(n_tokens, 'tokens,', n_types, 'types')

# 2.
print('Sentences:', chapter_one.count(???))

# 3.
total_length = 0
n_words = 0
for w in chapter_one:
    if w == 'eol':
        continue                 # skip to the next word
    total_length += ???          # the number of characters in w
    n_words += 1
print('Average word length:', round(total_length / n_words, 2))

**Activity 8 (word probabilities)**

Word probabilities work just like character probabilities. P(the) is given as an example.

1. Work out P(old), the probability of the word `old`.
2. Work out P(old | the): the number of times `the old` occurs divided by the number of times `the` occurs. Remember that bigram keys are written with a space, like `'the old'`.
3. Fill in the blank in the loop, which prints every word that follows `the` at least 5 times, with P(word | the). Which words are most likely after `the`?

In [ ]:
p_the = unigrams['the'] / total_unigrams
print('P(the) =', round(p_the, 4))

# 1.
p_old = ???
print('P(old) =', round(p_old, 4))

# 2.
p_old_given_the = ???
print('P(old | the) =', round(p_old_given_the, 4))

# 3.
for pair, count in bigrams.items():
    first, second = pair.split(' ')
    if first == 'the' and count >= 5:
        print(second, round(count / ???, 3))

And the counts can be used to assign probabilities to sentences in exactly the same way:

In [ ]:
sentence = ["a", "man", "was", "in", "the", "house"]
log_prob = 0.0
for i in range(len(sentence) - 1):
    pair = " ".join(sentence[i:i+2])
    p = bigrams.get(pair, 0) / unigrams[sentence[i]]    # P(next word | this word)
    log_prob += -math.log(p, 2)
log_prob

**Activity 9 (the first word of a sentence)**

Because we are using bigrams the first element for which we calculate the probability for is the second word. However to give the probability for the full sentence we want to also consider the first word. Update the code so that it does this. You may want to make reference to the lecture slides for the week in order to remind yourself of how we do this.

In [ ]:
# 'eol' marks the end of a sentence, so it can also stand for the START of the next one.
# Add 'eol' to the beginning of the list, so that the first word also gets a probability.
# (We use a sentence starting with 'he': no sentence in chapter one starts with 'a'.)
sentence = ["he", "was", "in", "the", "house"]
log_prob = 0.0
for i in range(len(sentence) - 1):
    pair = " ".join(sentence[i:i+2])
    p = bigrams.get(pair, 0) / unigrams[sentence[i]]
    log_prob += -math.log(p, 2)
log_prob

Now try calculating the log probability for a different sentence of your own creation

In [ ]:
sentence = []    # put your own words in the list
log_prob = 0.0
for i in range(len(sentence) - 1):
    pair = " ".join(sentence[i:i+2])
    p = bigrams.get(pair, 0) / unigrams[sentence[i]]
    log_prob += -math.log(p, 2)
log_prob

This probably won't work because of unseen bigrams: a pair of words that never occurs in the chapter gets probability 0, and taking the log of 0 gives an error. (A word that never occurs at all gives a `KeyError`.)

We need to solve this problem via smoothing.

For example, add-one smoothing:

In [ ]:
sentence = ["a", "man", "was", "in", "the", "house", "when", "the", "day", "arrived"]
vocab_size = len(unigrams)          # the number of different words
log_prob = 0.0
for i in range(len(sentence) - 1):
    pair = " ".join(sentence[i:i+2])
    # add one to every bigram count, and add the vocabulary size to the denominator
    p = (bigrams.get(pair, 0) + 1) / (unigrams.get(sentence[i], 0) + vocab_size)
    log_prob += -math.log(p, 2)
log_prob

Or alternatively, smoothing by interpolation, which mixes the bigram probability with the unigram probability of the next word:

In [ ]:
sentence = ["a", "man", "was", "in", "the", "house", "when", "the", "day", "arrived"]
# These lambdas can change but should sum to 1
lambda1 = 0.5
lambda2 = 1 - lambda1
log_prob = 0.0
for i in range(len(sentence) - 1):
    pair = " ".join(sentence[i:i+2])
    p_bigram = bigrams.get(pair, 0) / unigrams[sentence[i]]        # P(next word | this word)
    p_unigram = unigrams.get(sentence[i+1], 0) / total_unigrams    # P(next word)
    # combine the two, weighting each by its lambda
    p = lambda1 * p_bigram + lambda2 * p_unigram
    log_prob += -math.log(p, 2)
log_prob

### Putting it all together

**Activity 10 (scoring your own sentences)**

1. Replace each `???` with a sentence of your own (in lower case, without punctuation). The code writes them to a file, one per line.
2. The second part reads the file back and scores each sentence with the add-one smoothed model. Fill in the one blank: turn `line` into a list of words by making it lower case and then splitting it.
3. Run the cell. Which sentence does the model find least surprising (lowest perplexity)? Does a longer sentence always have a bigger total surprisal (`lp`)? Does it have a bigger perplexity?

In [ ]:
# 1. write your sentences to a file
my_sentences = ["the man was in the house",
                "???",
                "???",
                "???"]
with open('my_sentences.txt', 'w') as fp:
    for s in my_sentences:
        fp.write(s + '\n')

# 2. read them back and score each one
vocab_size = len(unigrams)
with open('my_sentences.txt') as fp:
    for line in fp:
        words = ???
        lp = 0.0
        for i in range(len(words) - 1):
            pair = " ".join(words[i:i+2])
            lp += -math.log((bigrams.get(pair, 0) + 1) / (unigrams.get(words[i], 0) + vocab_size), 2)
        perplexity = 2 ** (lp / (len(words) - 1))
        print(line.strip(), '|', round(lp, 2), '|', round(perplexity, 2))

⭐ **Challenge (optional) · Activity 11 (generating text)**

A language model can also *generate* text. Starting from the sentence-start token `eol`, this code repeatedly picks the **most likely** next word: the follower of the current word with the highest bigram count. This is called *greedy* generation.

Fill in the three blanks and run it. What happens? Why does greedy generation get stuck repeating itself? How might choosing the next word *at random*, in proportion to its probability, help?

In [ ]:
current = 'eol'
generated = []
for step in range(20):
    best_word = ''
    best_count = 0
    for pair, count in bigrams.items():
        first, second = pair.split(' ')
        if first == current and count > best_count:   # a better follower than any so far
            best_word = second
            best_count = count
    if best_count == 0:        # dead end: no word ever follows this one
        break
    generated.append(best_word)
    current = best_word              # move on to the word we just chose

print(' '.join(generated))

---
# Part 2 · Vector-based models of word meaning

This part looks at vector-based models of word meaning. A **vector** is just a list of numbers, so in Python we can store one as an ordinary list. Indexing and slicing work as for any list:

In [ ]:
a = [9, 2, 3, 5]
print(a[1])
print(a[1:3])

For vectors we can perform the operations that we learned about in our lecture. They work **element by element**: the first element of `a` with the first element of `b`, the second with the second, and so on. We use an index-based `for` loop, as in the Week 3 Python workbook.

Careful: `a + b` on two lists does **not** add them as vectors. It joins them into one longer list (try it!).

In [ ]:
# Vector addition
a = [9, 2, 3, 5]
b = [1, 2, 3, 4]
c = []
for i in range(len(a)):
    c.append(a[i] + b[i])
print(a)
print(b)
print(c)

In [ ]:
# Vector subtraction
a = [9, 2, 3, 5]
b = [1, 2, 3, 4]
c = []
for i in range(len(a)):
    c.append(a[i] - b[i])
print(a)
print(b)
print(c)

In [ ]:
# Dot product: multiply the vectors element by element, then add up the results
a = [9, 2, 3, 5]
b = [1, 2, 3, 4]
c = []
for i in range(len(a)):
    c.append(a[i] * b[i])
dp = sum(c)
print(a)
print(b)
print(c)
print(dp)

**Activity 12 (cosine)**

The cosine of the angle between two vectors is their dot product divided by the product of their lengths:

$$\cos(a, b) = \frac{a \cdot b}{\|a\| \, \|b\|} \qquad \text{where } \|a\| = \sqrt{a \cdot a}$$

The loop adds up three dot products at once: `a` with `b`, `a` with itself, and `b` with itself. Fill in the blanks. For a square root, raise a number to the power 0.5: `x ** 0.5`.

⭐ **Challenge (optional):** change `b` so that it is the same as `a`, and run the cell again. A vector always has cosine 1 with itself.

In [ ]:
a = [9, 2, 3, 5]
b = [1, 2, 3, 4]

dot = 0          # a . b
a_squares = 0    # a . a
b_squares = 0    # b . b
for i in range(len(a)):
    dot += ???                  # the element of a at position i, times the element of b at position i
    a_squares += ???
    b_squares += ???

length_a = ???                  # the square root of a . a
length_b = ???
cos_ab = ???                    # the dot product divided by the product of the two lengths
print(cos_ab)

### Building Word Vectors

In [ ]:
# We downloaded the book at the start of Part 1. This time we use the whole novel, not just chapter one.
with open('2554-0.txt') as fp:
    c_and_p = fp.read()
# keep the novel itself: from the first chapter heading up to Project Gutenberg's end-of-book notice
start = c_and_p.find('CHAPTER I\n')
end = c_and_p.find('*** END OF THE PROJECT GUTENBERG')
text = c_and_p[start:end].lower().replace('\n', ' ')
# keep only the letters a-z and spaces (as in Part 1), then split the text into a list of words
allowed = 'abcdefghijklmnopqrstuvwxyz '
kept = []
for ch in text:
    if ch in allowed:
        kept.append(ch)
words = ''.join(kept).split()

In [ ]:
words[:10]

**Activity 13 (to be tackled together): counting contexts**

To build a vector for each word, we count how often each *context* word occurs within `window_size` words of it.

* Only the 1000 most frequent words are used as contexts. Rare context words add little information. To find them, we count every word with a dictionary, make a list of `(count, word)` pairs and sort it from largest to smallest, as in the ⭐ Challenge in the Week 2 seminar. We also keep them in a set, `context_set`, because checking whether something is in a set is very fast.
* The counts go in `M`, a **dictionary of dictionaries**. `M[word]` is itself a counting dictionary, mapping each context word to how often it occurs near `word`. So `M["walk"]["the"]` is the number of times `the` occurs within two words of `walk`.
* For each position `i` in the text, the inner loop visits the positions `j` around it. The first conditions check that `j` is a real position in the text and is not `i` itself. Fill in the last condition (the word at position `j` must be one of our contexts), then add one to its count using the counting pattern.

In [ ]:
token_count = len(words)

# The contexts: the 1000 most frequent words.
# Count every word, make a list of (count, word) pairs, sort it from largest to smallest, then keep the first 1000 words.
word_counts = {}
for word in words:
    word_counts[word] = word_counts.get(word, 0) + 1
pairs = []
for word, count in word_counts.items():
    pairs.append((count, word))
pairs = sorted(pairs, reverse=True)
context_list = []
for count, word in pairs[:1000]:
    context_list.append(word)
context_set = set(context_list)

# M[word][context] counts how often context occurs within window_size words of word
M = {}
window_size = 2

for i in range(token_count):
    word = words[i]
    if word not in M:
        M[word] = {}          # first time we have seen this word: give it an empty counting dictionary
    for j in range(i - window_size, i + window_size + 1):
        if j >= 0 and j < token_count and j != i and ???:   # ... and the word at position j is a context word
            M[word][???] = M[word].get(???, 0) + 1

In [ ]:
M["walk"]

**Activity 14**

Now we can turn the counts into vectors. The vector for a word has one number for each word in `context_list`, in the same order: how often that context word occurs near it (0 if it never does). The first loop builds the vector for "walk" as an example.

Use the cosine calculation from Activity 12 to compare "walk" with "run", and "walk" with "shine". Fill in the blanks. What does the outcome tell us?

⭐ **Challenge (optional):** add more words to `others`. Which words come out as most similar to "walk"?

In [ ]:
walk_vec = []
for context in context_list:
    walk_vec.append(M["walk"].get(context, 0))     # 0 if this context never occurs near "walk"

others = ["run", "shine"]
for other in others:
    other_vec = []
    for context in context_list:
        other_vec.append(???)                      # the count for this context near other

    dot = 0
    walk_squares = 0
    other_squares = 0
    for i in range(len(context_list)):
        dot += ???
        walk_squares += ???
        other_squares += ???
    print("walk ~", other, ":", ???)               # remember: square root is ** 0.5